# S06 — Balanced search trees
Step-by-step lab. Run each cell. TODOs are intentional; no graded assignment answers.

## 1. Node, height and rotation
Height(empty)=0. BF = left height − right height. Read the right rotation, then complete its mirror.

In [ ]:
from dataclasses import dataclass
@dataclass
class Node:
    key: int
    left: object = None
    right: object = None
    height: int = 1

def h(node):
    return node.height if node else 0

def update(node):
    node.height = 1 + max(h(node.left), h(node.right))

def bf(node):
    return h(node.left) - h(node.right)

def rotate_right(y):
    x = y.left
    y.left = x.right
    x.right = y
    update(y)
    update(x)
    return x

def rotate_left(x):
    # TODO: mirror rotate_right; preserve middle subtree
    raise NotImplementedError

def insert(root, key):
    if root is None:
        return Node(key)
    if key < root.key:
        root.left = insert(root.left, key)
    elif key > root.key:
        root.right = insert(root.right, key)
    else:
        return root
    update(root)
    # TODO: classify and repair LL, RR, LR, RL; return new root
    return root

def inorder(root):
    return inorder(root.left) + [root.key] + inorder(root.right) if root else []

def validate(root, lo=float('-inf'), hi=float('inf')):
    if root is None:
        return 0
    assert lo < root.key < hi, 'BST ordering'
    a = validate(root.left, lo, root.key)
    b = validate(root.right, root.key, hi)
    assert root.height == 1 + max(a,b), 'stale height'
    assert abs(a-b) <= 1, 'AVL balance'
    return root.height

def build(keys):
    root = None
    for key in keys:
        root = insert(root, key)
    return root

if __name__ == '__main__':
    y = Node(30, Node(20, Node(10)))
    update(y.left); update(y)
    root = rotate_right(y)
    assert inorder(root) == [10,20,30]
    assert validate(root) == 2
    print('Right rotation demo passed; student TODOs remain.')


## 2. Implement insertion
After standard BST insertion, update cached height. If BF > 1, inspect left child; if BF < −1, inspect right child. Use one or two rotations. Return the new subtree root. Rerun the definition cell after editing.

In [ ]:
for seq in ([30,20,10],[10,20,30],[30,10,20],[10,30,20]):
    root = build(seq)
    try:
        assert root.key == 20
        assert inorder(root) == sorted(seq)
        validate(root)
        print(seq, "PASS")
    except (AssertionError, NotImplementedError):
        print(seq, "TODO: complete insertion")

## 3. Edge cases
Run after completing insertion. Validate ordering, height and balance after each insertion.

In [ ]:
import random
assert validate(None) == 0
for seq in ([7], [7,7], list(range(100)), random.Random(42).sample(range(100),100)):
    root = None
    try:
        for k in seq:
            root = insert(root,k)
            validate(root)
        assert inorder(root) == sorted(set(seq))
        print(len(seq), "PASS")
    except (AssertionError, NotImplementedError):
        print(len(seq), "TODO: repair implementation")

## 4. Compare plain BST height
The following baseline is an iterative unbalanced BST. Compare identical inputs and report height, not a universal timing guarantee.

In [ ]:
def plain_height(keys):
    tree = {}
    root = None
    maximum = 0
    for key in keys:
        if root is None:
            root = key; tree[key] = [None,None]; maximum = 1; continue
        cur, depth = root, 1
        while key != cur:
            side = int(key > cur)
            if tree[cur][side] is None:
                tree[cur][side] = key; tree[key] = [None,None]
                maximum = max(maximum, depth+1); break
            cur = tree[cur][side]; depth += 1
    return maximum
for size in (20,100,500):
    for label, keys in [("sorted",list(range(size))), ("shuffled",random.Random(42).sample(range(size),size))]:
        try:
            avl = validate(build(keys))
        except (AssertionError,NotImplementedError):
            avl = "TODO"
        print(size,label,plain_height(keys),avl)

## 5. Other tree families
- Red-black: black root 20, red children 10 and 30. Insert red 5; identify violation and recolouring.
- Splay: trace access to 10 in 30→20→10. Distinguish zig-zig and zig-zag.
- B-tree: t=2, full root [10,20,30]. Split before inserting 25. Count page visits.

## Exit ticket
Choose a tree for a disk index and one for repeated hot-key access; explain the trade-offs.